# NB_lab_test_trends

Builds `adb_caresync.gold.lab_test_trends` (one row per test_name per calendar month) from `silver.lab_results` per the STTM `sttm_silver_to_gold.xlsx`, with calendar month/quarter names and a month-over-month abnormal-rate trend via LAG(). Full rebuild on every run.

In [0]:
%sql
-- Full rebuild: one row per test_name per year/month (self-contained aggregation; optional calendar LEFT JOIN)
CREATE OR REPLACE TABLE adb_caresync.gold.lab_test_trends
COMMENT 'Gold: one row per lab test type per month with volumes, abnormal/critical rates and month-over-month abnormal-rate trend. Source: STTM silver->gold.'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                            AS year,
    month(test_date)                           AS month,
    COUNT(*)                                   AS total_tests,
    COUNT_IF(result_status = 'Abnormal')       AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')       AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
with_rates AS (
  SELECT
    m.*,
    CAST(ROUND(m.abnormal_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(m.critical_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_trend AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM with_rates r
)
SELECT
  t.test_name                                                                  AS test_name,
  CAST(t.year AS INT)                                                          AS year,
  CAST(t.month AS INT)                                                         AS month,
  c.month_name                                                                 AS month_name,
  c.quarter_name                                                               AS quarter_name,
  CAST(t.total_tests AS INT)                                                   AS total_tests,
  CAST(t.abnormal_test_count AS INT)                                           AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                           AS critical_test_count,
  t.abnormal_rate_pct                                                          AS abnormal_rate_pct,
  t.critical_rate_pct                                                          AS critical_rate_pct,
  CAST(t.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                         AS prev_month_abnormal_rate_pct,
  CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2))   AS abnormal_rate_change_pct   -- NULL when no prior month
FROM with_trend t
LEFT JOIN adb_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(make_date(t.year, t.month, 1), 'yyyyMMdd') AS INT);

In [0]:
%sql
-- Grain must be unique on (test_name, year, month); test totals must reconcile with silver.lab_results
SELECT
  COUNT(*)                                                     AS gold_rows,
  COUNT(DISTINCT test_name, year, month)                       AS distinct_grain,
  SUM(total_tests)                                             AS gold_total_tests,
  (SELECT COUNT(*) FROM adb_caresync.silver.lab_results)       AS silver_total_tests,
  COUNT(DISTINCT test_name)                                    AS test_types,
  COUNT_IF(month_name IS NULL)                                 AS missing_calendar_match,
  COUNT_IF(prev_month_abnormal_rate_pct IS NULL)               AS first_month_rows
FROM adb_caresync.gold.lab_test_trends;

In [0]:
# Return the gold table's record count to the caller (e.g. a job task or dbutils.notebook.run)
updated_df = spark.table("adb_caresync.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))